In [ ]:
import pandas as pd
import numpy as np
import scipy
import scipy.sparse as sp
import scipy.io as sio
import scipy.stats as stats
from tqdm.notebook import tqdm


import os
os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from plotnine import *

import matplotlib.pyplot as plt 

import pickle

from joblib import Parallel, delayed

import sys
from pathlib import Path

# Get project root as parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.methods.GWASH_funcs import *
from src.methods.GWASH_sim_funcs import *
from src.methods.ldsc_barebones import *

from src.simtools.sim_utils import *
from src.simtools.data_generation import data_generation as data_generation
from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
from src.simtools.do_analysis import do_analysis as do_analysis
from src.simtools.run_simulations import run_simulations as run_simulations
from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel
from src.simtools.visualization import visualize

from natsort import natsorted

import pickle

seed_num = 123
np.random.seed(seed_num)

os.chdir(PROJECT_ROOT)

# AR1

In [ ]:
#n,m,num_sims = int(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3])
n,m,num_sims = 5000,10000,100
seed_num = 123
# Make Save Folder
nb_name_prefix = 'ldsc_pop_sim_overleaf_class-PCA-ENABLED_sstats_methods_only_n{n}_m{m}_sims{num_sims}'.format(n=str(n),m = str(m),num_sims = str(num_sims)) # name of overall project


current_dateTime = datetime.now()
date_name = 'seed'+str(seed_num)

save_dir = 'save_data/' + nb_name_prefix +'/' + date_name + '/'
raw_data_path = save_dir +'/raw_data/'
overleaf_path = save_dir+'/overleaf_tables/'
make_path(save_dir)
make_path(raw_data_path)
make_path(overleaf_path)

# Think about what parameters am I inputting:
# Properties of X that can be tweaked: n,m, sigma_s, rho1, rho2, Fst
# LD Matrix properties (if an actual LD matrix is used, its path, if a reference ldscore panel is generated): realistic, prefix,make_ref_ldscores
# Method properties (related to methods used): reml_tol, reml_max_iters
# Statistical properties: extra things like principal components regression: nPCs,regress_PC_out, regress_X_on_PC, regress_y_on_PC
# Simulation Properties (related to running the actual simulations): num_sims
# Debugging Properties (use this to actual check if the shortcuts we use are correct): calc_mu_hat_2_fast,old



X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0,'rho2':None,'Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':False}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':num_sims,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress': True, 'run_gcta':False}

list_of_dicts = [X_properties,ld_mat_properties,ref_X_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['non_realistic'] = params


sim_key = 'non_realistic'

locals().update(to_run[sim_key])

res_dict = dict()
res_dict_raw = dict()

#Fsts = [0,0.01,0.03,0.05,0.07,0.1]
counter = -1
multithreading = True

sigma_s = 0
rho1 = 0.995
rho2 = None
Fst = 0
m_causal = None
h2_pop = 0.2
fixed_m_causal = False

counter += 1
np.random.seed(counter)
key = str(Fst)

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = ref_pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None

ldsc_h2s = []
true_fves = []
AR1_ldscores = np.zeros((num_sims,m))
u2s = np.zeros((num_sims,m))
for q in tqdm(range(num_sims)):
    np.random.seed(seed_num + q)
    
    my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat,do_ldscore_bias_correct = True)
    #res_dfs = run_simulations(my_data_gen,heels_max_iters = 999999,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress,run_gcta = run_gcta).num_sims_simulations_publication()
    
    
    X,b = my_data_gen.gen_X(realistic = realistic)
    y = my_data_gen.gen_y(X,b)
    
    my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = 5,regress_X_on_PC = False,regress_y_on_PC = False)
    if False:
        regress_res = my_data_preprocessing.regress_out_PCs(X,y)
        X = regress_res['X_res']
        y = regress_res['y_res']
        
    h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
    true_fves.append(h2_samp_fve)
    if True:
        X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
    else:
        X_tilde = X
    if True:
        y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
    else:
        y_tilde = y
    # if self.data_gen.ref_ldscores is None, then generate ldscores. Otherwise, use self.data_gen.ref_ldscores
    
    u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)
    AR1_ldscores[q,:] = ldscores.flatten()
    u2s[q,:] = u2.flatten()
    M = my_data_gen.M
    N_per_SNP = my_data_gen.N_per_SNP
    ldsc_reg_weights = ldscores
    
    ldsc_h2, ldsc_se,ldsc_intercept,weights = do_ldsc_regression(u2,ldscores,M,N_per_SNP,ldsc_reg_weights,intercept = None)

    ldsc_h2s.append(ldsc_h2)
    
param_dict = {'rho':rho,'Fst':Fst,'sigma_s':sigma_s}
res_dict = {'AR1_ldscores':AR1_ldscores,'u2s':u2s,'ldsc_h2s':ldsc_h2s,'true_fves':true_fves,params = param_dict}

with open("save_data/appendix/ldscores_chi2_100sim_data/AR1/AR1_rho{rho}_Fst{Fst}_sigma_s{sigma_s}_ldscores_u2s_100sims_no_refpanel_seed123.pkl".format(rho = str(rho).replace('.',''),Fst = str(Fst).replace('.',''),sigma_s =str(sigma_s).replace('.','')), "wb") as f:
        pickle.dump(res_dict, f)

In [ ]:
#n,m,num_sims = int(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3])
n,m,num_sims = 5000,10000,100
seed_num = 123
# Make Save Folder
nb_name_prefix = 'ldsc_pop_sim_overleaf_class-PCA-ENABLED_sstats_methods_only_n{n}_m{m}_sims{num_sims}'.format(n=str(n),m = str(m),num_sims = str(num_sims)) # name of overall project


current_dateTime = datetime.now()
date_name = 'seed'+str(seed_num)

save_dir = 'save_data/' + nb_name_prefix +'/' + date_name + '/'
raw_data_path = save_dir +'/raw_data/'
overleaf_path = save_dir+'/overleaf_tables/'
make_path(save_dir)
make_path(raw_data_path)
make_path(overleaf_path)

# Think about what parameters am I inputting:
# Properties of X that can be tweaked: n,m, sigma_s, rho1, rho2, Fst
# LD Matrix properties (if an actual LD matrix is used, its path, if a reference ldscore panel is generated): realistic, prefix,make_ref_ldscores
# Method properties (related to methods used): reml_tol, reml_max_iters
# Statistical properties: extra things like principal components regression: nPCs,regress_PC_out, regress_X_on_PC, regress_y_on_PC
# Simulation Properties (related to running the actual simulations): num_sims
# Debugging Properties (use this to actual check if the shortcuts we use are correct): calc_mu_hat_2_fast,old



X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0,'rho2':None,'Fst':0}
ld_mat_properties = {'realistic': False,'prefix': None,'make_ref_ldscores':True}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':num_sims,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress': True, 'run_gcta':False}

list_of_dicts = [X_properties,ld_mat_properties,ref_X_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['non_realistic'] = params


sim_key = 'non_realistic'

locals().update(to_run[sim_key])

res_dict = dict()
res_dict_raw = dict()

#Fsts = [0,0.01,0.03,0.05,0.07,0.1]
counter = -1
multithreading = True

sigma_s = 0
rho1 = 0.995
rho2 = None
Fst = 0
m_causal = None
h2_pop = 0.2
fixed_m_causal = False

counter += 1
np.random.seed(counter)
key = str(Fst)

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = ref_pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None

ldsc_h2s = []
true_fves = []
AR1_ldscores = np.zeros((num_sims,m))
u2s = np.zeros((num_sims,m))
for q in tqdm(range(num_sims)):
    np.random.seed(seed_num + q)
    
    my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat,do_ldscore_bias_correct = True)
    #res_dfs = run_simulations(my_data_gen,heels_max_iters = 999999,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress,run_gcta = run_gcta).num_sims_simulations_publication()
    
    
    X,b = my_data_gen.gen_X(realistic = realistic)
    y = my_data_gen.gen_y(X,b)
    
    my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = 5,regress_X_on_PC = False,regress_y_on_PC = False)
    if False:
        regress_res = my_data_preprocessing.regress_out_PCs(X,y)
        X = regress_res['X_res']
        y = regress_res['y_res']
        
    h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
    true_fves.append(h2_samp_fve)
    if True:
        X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
    else:
        X_tilde = X
    if True:
        y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
    else:
        y_tilde = y
    # if self.data_gen.ref_ldscores is None, then generate ldscores. Otherwise, use self.data_gen.ref_ldscores
    
    u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)
    AR1_ldscores[q,:] = ldscores.flatten()
    u2s[q,:] = u2.flatten()
    M = my_data_gen.M
    N_per_SNP = my_data_gen.N_per_SNP
    ldsc_reg_weights = ldscores
    
    ldsc_h2, ldsc_se,ldsc_intercept,weights = do_ldsc_regression(u2,ldscores,M,N_per_SNP,ldsc_reg_weights,intercept = None)

    ldsc_h2s.append(ldsc_h2)
    
param_dict = {'rho':rho,'Fst':Fst,'sigma_s':sigma_s}
res_dict = {'AR1_ldscores':AR1_ldscores,'u2s':u2s,'ldsc_h2s':ldsc_h2s,'true_fves':true_fves,params = param_dict}

with open("save_data/appendix/ldscores_chi2_100sim_data/AR1/AR1_rho{rho}_Fst{Fst}_sigma_s{sigma_s}_ldscores_u2s_100sims_refpanel_seed123.pkl".format(rho = str(rho).replace('.',''),Fst = str(Fst).replace('.',''),sigma_s =str(sigma_s).replace('.','')), "wb") as f:
        pickle.dump(res_dict, f)

# Realistic

In [ ]:
#n,m,num_sims = int(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3])
n,m,num_sims = 5000,10000,100
seed_num = 123
# Make Save Folder
nb_name_prefix = 'ldsc_pop_sim_overleaf_class-PCA-ENABLED_sstats_methods_only_n{n}_m{m}_sims{num_sims}'.format(n=str(n),m = str(m),num_sims = str(num_sims)) # name of overall project


current_dateTime = datetime.now()
date_name = 'seed'+str(seed_num)

save_dir = 'save_data/' + nb_name_prefix +'/' + date_name + '/'
raw_data_path = save_dir +'/raw_data/'
overleaf_path = save_dir+'/overleaf_tables/'
make_path(save_dir)
make_path(raw_data_path)
make_path(overleaf_path)

# Think about what parameters am I inputting:
# Properties of X that can be tweaked: n,m, sigma_s, rho1, rho2, Fst
# LD Matrix properties (if an actual LD matrix is used, its path, if a reference ldscore panel is generated): realistic, prefix,make_ref_ldscores
# Method properties (related to methods used): reml_tol, reml_max_iters
# Statistical properties: extra things like principal components regression: nPCs,regress_PC_out, regress_X_on_PC, regress_y_on_PC
# Simulation Properties (related to running the actual simulations): num_sims
# Debugging Properties (use this to actual check if the shortcuts we use are correct): calc_mu_hat_2_fast,old



X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0,'rho2':None,'Fst':0}
ld_mat_properties = {'realistic': True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':False}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':num_sims,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress': True, 'run_gcta':False}

list_of_dicts = [X_properties,ld_mat_properties,ref_X_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['realistic'] = params


sim_key = 'realistic'

locals().update(to_run[sim_key])

res_dict = dict()
res_dict_raw = dict()

#Fsts = [0,0.01,0.03,0.05,0.07,0.1]
counter = -1
multithreading = True

sigma_s = 0
rho1 = 0.995
rho2 = None
Fst = 0
m_causal = None
h2_pop = 0.2
fixed_m_causal = False

counter += 1
np.random.seed(counter)
key = str(Fst)

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = ref_pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None

ldsc_h2s = []
true_fves = []
AR1_ldscores = np.zeros((num_sims,m))
u2s = np.zeros((num_sims,m))
for q in tqdm(range(num_sims)):
    np.random.seed(seed_num + q)
    
    my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat,do_ldscore_bias_correct = True)
    #res_dfs = run_simulations(my_data_gen,heels_max_iters = 999999,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress,run_gcta = run_gcta).num_sims_simulations_publication()
    
    
    X,b = my_data_gen.gen_X(realistic = realistic)
    y = my_data_gen.gen_y(X,b)
    
    my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = 5,regress_X_on_PC = False,regress_y_on_PC = False)
    if False:
        regress_res = my_data_preprocessing.regress_out_PCs(X,y)
        X = regress_res['X_res']
        y = regress_res['y_res']
        
    h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
    true_fves.append(h2_samp_fve)
    if True:
        X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
    else:
        X_tilde = X
    if True:
        y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
    else:
        y_tilde = y
    # if self.data_gen.ref_ldscores is None, then generate ldscores. Otherwise, use self.data_gen.ref_ldscores
    
    u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)
    AR1_ldscores[q,:] = ldscores.flatten()
    u2s[q,:] = u2.flatten()
    M = my_data_gen.M
    N_per_SNP = my_data_gen.N_per_SNP
    ldsc_reg_weights = ldscores
    
    ldsc_h2, ldsc_se,ldsc_intercept,weights = do_ldsc_regression(u2,ldscores,M,N_per_SNP,ldsc_reg_weights,intercept = None)

    ldsc_h2s.append(ldsc_h2)
    
param_dict = {'rho':rho,'Fst':Fst,'sigma_s':sigma_s}
res_dict = {'AR1_ldscores':AR1_ldscores,'u2s':u2s,'ldsc_h2s':ldsc_h2s,'true_fves':true_fves,params = param_dict}

with open("save_data/appendix/ldscores_chi2_100sim_data/realistic/realistic_Fst{Fst}_sigma_s{sigma_s}_ldscores_u2s_100sims_no_refpanel_seed123.pkl".format(rho = str(rho).replace('.',''),Fst = str(Fst).replace('.',''),sigma_s =str(sigma_s).replace('.','')), "wb") as f:
        pickle.dump(res_dict, f)

In [ ]:
#n,m,num_sims = int(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3])
n,m,num_sims = 5000,10000,100
seed_num = 123
# Make Save Folder
nb_name_prefix = 'ldsc_pop_sim_overleaf_class-PCA-ENABLED_sstats_methods_only_n{n}_m{m}_sims{num_sims}'.format(n=str(n),m = str(m),num_sims = str(num_sims)) # name of overall project


current_dateTime = datetime.now()
date_name = 'seed'+str(seed_num)

save_dir = 'save_data/' + nb_name_prefix +'/' + date_name + '/'
raw_data_path = save_dir +'/raw_data/'
overleaf_path = save_dir+'/overleaf_tables/'
make_path(save_dir)
make_path(raw_data_path)
make_path(overleaf_path)

# Think about what parameters am I inputting:
# Properties of X that can be tweaked: n,m, sigma_s, rho1, rho2, Fst
# LD Matrix properties (if an actual LD matrix is used, its path, if a reference ldscore panel is generated): realistic, prefix,make_ref_ldscores
# Method properties (related to methods used): reml_tol, reml_max_iters
# Statistical properties: extra things like principal components regression: nPCs,regress_PC_out, regress_X_on_PC, regress_y_on_PC
# Simulation Properties (related to running the actual simulations): num_sims
# Debugging Properties (use this to actual check if the shortcuts we use are correct): calc_mu_hat_2_fast,old



X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0,'rho2':None,'Fst':0}
ld_mat_properties = {'realistic': True,'prefix': '1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':num_sims,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress': True, 'run_gcta':False}

list_of_dicts = [X_properties,ld_mat_properties,ref_X_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['realistic'] = params


sim_key = 'realistic'

locals().update(to_run[sim_key])

res_dict = dict()
res_dict_raw = dict()

#Fsts = [0,0.01,0.03,0.05,0.07,0.1]
counter = -1
multithreading = True

sigma_s = 0
rho1 = 0.995
rho2 = None
Fst = 0
m_causal = None
h2_pop = 0.2
fixed_m_causal = False

counter += 1
np.random.seed(counter)
key = str(Fst)

if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = ref_sigma_s,h2_pop = h2_pop,pm_causal = ref_pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None

ldsc_h2s = []
true_fves = []
AR1_ldscores = np.zeros((num_sims,m))
u2s = np.zeros((num_sims,m))
for q in tqdm(range(num_sims)):
    np.random.seed(seed_num + q)
    
    my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat,do_ldscore_bias_correct = True)
    #res_dfs = run_simulations(my_data_gen,heels_max_iters = 999999,seed = seed_num,scaleX = scaleX,scaley = scaley,num_sims = num_sims,nPCs = nPCs,regress_PC_out = regress_PC_out,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC, multithreading = multithreading, n_jobs = n_jobs, realistic = realistic, calc_mu_hat_2_fast = calc_mu_hat_2_fast,reml_tol = reml_tol,track_progress = track_progress,run_gcta = run_gcta).num_sims_simulations_publication()
    
    
    X,b = my_data_gen.gen_X(realistic = realistic)
    y = my_data_gen.gen_y(X,b)
    
    my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = 5,regress_X_on_PC = False,regress_y_on_PC = False)
    if False:
        regress_res = my_data_preprocessing.regress_out_PCs(X,y)
        X = regress_res['X_res']
        y = regress_res['y_res']
        
    h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
    true_fves.append(h2_samp_fve)
    if True:
        X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
    else:
        X_tilde = X
    if True:
        y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
    else:
        y_tilde = y
    # if self.data_gen.ref_ldscores is None, then generate ldscores. Otherwise, use self.data_gen.ref_ldscores
    
    u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)
    AR1_ldscores[q,:] = ldscores.flatten()
    u2s[q,:] = u2.flatten()
    M = my_data_gen.M
    N_per_SNP = my_data_gen.N_per_SNP
    ldsc_reg_weights = ldscores
    
    ldsc_h2, ldsc_se,ldsc_intercept,weights = do_ldsc_regression(u2,ldscores,M,N_per_SNP,ldsc_reg_weights,intercept = None)

    ldsc_h2s.append(ldsc_h2)
    
param_dict = {'rho':rho,'Fst':Fst,'sigma_s':sigma_s}
res_dict = {'AR1_ldscores':AR1_ldscores,'u2s':u2s,'ldsc_h2s':ldsc_h2s,'true_fves':true_fves,params = param_dict}

with open("save_data/appendix/ldscores_chi2_100sim_data/realistic/realistic_Fst{Fst}_sigma_s{sigma_s}_ldscores_u2s_100sims_refpanel_seed123.pkl".format(Fst = str(Fst).replace('.',''),sigma_s =str(sigma_s).replace('.','')), "wb") as f:
        pickle.dump(res_dict, f)